# Notebook 053: Distributed REST API and Webhook Ingestion without Driver OOM

**Module:** Module 05: Data Ingestion Patterns, Auto Loader & File Formats  
**Difficulty:** `Advanced`  
**Databricks Masterclass Reference:** Video 16: LakeFlow Connect & Video 25: Genie Code & Optimization  

---

## 1. Business Scenario & Real-World Context
A financial data vendor exposes historical equity prices via an authenticated REST API with rate limits. The engineering team must fetch data for 50,000 distinct stock tickers. Running loops on the driver node takes 18 hours or crashes the driver with out-of-memory errors.

## 2. Technical Requirement & Problem Statement
Implement parallel, distributed HTTP REST API ingestion in PySpark: partition entity IDs across worker nodes, use `mapPartitions` to maintain persistent connection sessions (`requests.Session`), apply batch chunking, handle exponential backoff rate limits, and assemble responses into DataFrames.

## 3. Spark & Databricks Architecture Notes
Using `mapPartitions` initializes one HTTP connection pool (`requests.Session()`) per partition rather than opening a new TCP connection per row. Distributing requests across 20 executor partitions achieves 20x concurrency while honoring per-node rate limit quotas.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_053_Distributed_REST_API_and_Webhook_Ingestion_without_Driver_OOM")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Preparing Target Partitioned Request Keys
Create target ticker symbols distributed across 4 partitions.

In [ ]:
tickers = [f"TICKER_{i:04d}" for i in range(1, 21)]
# Distribute across 4 partitions
df_tickers = spark.createDataFrame([(t,) for t in tickers], ["ticker"]).repartition(4)

print(f"Total Tickers: {df_tickers.count()} across {df_tickers.rdd.getNumPartitions()} partitions")

## 6. Distributed Ingestion via mapPartitions and Session Pooling
Execute parallel HTTP calls inside worker tasks.

In [ ]:
import time
import json

def fetch_ticker_data_partition(iterator):
    # Executed ONCE per partition on worker executor
    # Simulates reusable requests.Session()
    session_id = f"worker_session_{time.time()}"
    results = []
    
    for row in iterator:
        ticker = row.ticker
        # Simulate network latency to REST API (e.g. 5ms)
        time.sleep(0.005)
        # Mock REST API response payload
        mock_response = {
            "ticker": ticker,
            "price": round(150.0 + (hash(ticker) % 50), 2),
            "currency": "USD",
            "status_code": 200,
            "fetched_at": time.strftime("%Y-%m-%d %H:%M:%S")
        }
        results.append((
            mock_response["ticker"],
            mock_response["price"],
            mock_response["currency"],
            mock_response["status_code"],
            mock_response["fetched_at"]
        ))
    return iter(results)

# Execute parallel extraction across all executors
rdd_responses = df_tickers.rdd.mapPartitions(fetch_ticker_data_partition)
schema = "ticker STRING, price DOUBLE, currency STRING, http_status INT, fetched_at STRING"
df_api_results = spark.createDataFrame(rdd_responses, schema)

print("Distributed REST API Ingestion Results:")
df_api_results.show(10)

## 7. Validation & Status Code Auditing
Verify all requests completed with HTTP 200.

In [ ]:
assert df_api_results.count() == 20, "Ticker count mismatch"
assert df_api_results.filter(F.col("http_status") != 200).count() == 0, "Non-200 responses detected"
print("All distributed API calls verified successfully!")

## 9. Key Architectural Takeaways
- **`mapPartitions` allows initializing heavyweight objects (database connections, HTTP sessions) once per partition rather than per row.**
- **Distributing API calls across worker partitions speeds up extraction linearly with cluster worker count.**
- **Always include rate-limiting sleep intervals and exponential backoff retry logic inside worker partitions.**

## 10. Senior Data Engineering Interview Questions
1. Why is `mapPartitions` far superior to `map` for executing REST API requests or database lookups?
2. How do you handle API rate-limiting (HTTP 429 Too Many Requests) across multiple distributed Spark executors?
3. What is the risk of driver OutOfMemory (OOM) if you fetch millions of API responses using `.collect()`?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
